In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from utils import model_report

In [2]:
df = pd.read_csv('data/spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]

In [3]:
x = df['v2']
y = df['v1']
y_mapped = y.map({'ham' : 0, 'spam' : 1})

In [4]:
x_train, _x, y_train, _y = train_test_split(x, y_mapped, test_size=0.2, random_state=36)
x_cv, x_test, y_cv, y_test = train_test_split(_x, _y, test_size=0.5, random_state=36)


print(x_train.shape, y_train.shape, x_cv.shape, x_test.shape, y_cv.shape, y_test.shape)

(4457,) (4457,) (557,) (558,) (557,) (558,)


In [5]:
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')

x_train_dfidf = vectorizer.fit_transform(x_train)
x_cv_dfidf = vectorizer.transform(x_cv)
x_test_dfidf = vectorizer.transform(x_test)

In [6]:
# model = XGBClassifier(
#     n_estimators=300,
#     max_depth=6,
#     learning_rate=0.1,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     min_child_weight=1,
#     gamma=0,
#     reg_alpha=0,
#     reg_lambda=1,
#     objective='binary:logistic',
#     eval_metric='logloss',
#     random_state=36,
#     n_jobs=-1
# )

model = XGBClassifier()
model.load_model('models/pro1_logloss_067.json')

In [7]:
# history = model.fit(
#     x_train_dfidf, y_train,
#     eval_set=[(x_cv_dfidf, y_cv)],    
# )

# model.save_model("models/pro1_logloss_067.json")

In [8]:
# y_pred = model.predict(x_cv_dfidf)
y_prob_cv = model.predict_proba(x_cv_dfidf)[:, 1]
y_pred_cv = (y_prob_cv >= 0.3).astype(int)


In [9]:
model_report(y_cv, y_pred_cv)

Accuracy: 0.9820466786355476
Precision: 0.9529411764705882
Recall: 0.9310344827586207
F1: 0.9418604651162791

 ----------------------------------------

Confusion Matrix: 
 [[466   4]
 [  6  81]]

 ----------------------------------------

Classification Report: 
               precision    recall  f1-score   support

           0       0.99      0.99      0.99       470
           1       0.95      0.93      0.94        87

    accuracy                           0.98       557
   macro avg       0.97      0.96      0.97       557
weighted avg       0.98      0.98      0.98       557



In [11]:
y_test_prob = model.predict_proba(x_test_dfidf)[:, 1]
y_test_pred = (y_test_prob >= 0.3).astype(int)

model_report(y_test, y_test_pred)

Accuracy: 0.967741935483871
Precision: 0.9152542372881356
Recall: 0.8059701492537313
F1: 0.8571428571428571

 ----------------------------------------

Confusion Matrix: 
 [[486   5]
 [ 13  54]]

 ----------------------------------------

Classification Report: 
               precision    recall  f1-score   support

           0       0.97      0.99      0.98       491
           1       0.92      0.81      0.86        67

    accuracy                           0.97       558
   macro avg       0.94      0.90      0.92       558
weighted avg       0.97      0.97      0.97       558

